In [ ]:
import json
from os import path

import geopandas as gpd
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
import xvec  # noqa: F401
from IPython.display import display
from meteora import clients
from skops import io

# Update this list with the output of io.get_untrusted_types() from the train notebook
SKOPS_TRUSTED = [
    "interpret.glassbox._ebm._ebm.ExplainableBoostingRegressor",
    "interpret.glassbox._ebm._bin.EBMPreprocessor",
    "swiss_uhi_lcd.regr_utils.BestScaleRadiationTransformer",
]

sns.set_style("whitegrid")

In [ ]:
aws_ts_cube_filepath = "../data/interim/bern-aws-ts-cube.nc"
lcd_ts_df_filepath = "../data/interim/bern-lcd-ts-df.csv"
lcd_stations_gdf_filepath = "../data/interim/bern-lcd-stations.gpkg"

# model_filepath = f"../models/{station_model}-gam-ar.pickle"
models_dir = "../models"
station_model = "Decentlab"
station_model_dict_filepath = "../data/processed/station-model-dict.json"

# output file
dst_ts_df_filepath = "../data/interim/bern-cor-ts-df.csv"

# for reporting
parallel_ts_df_filepath = "../data/raw/parallel-2025-int.csv"
validation_region = "Zollikofen"

In [ ]:
# aws_ts_df = pd.read_csv(aws_ts_df_filepath, parse_dates=["time"]).set_index("time")
lcd_ts_df = pd.read_csv(lcd_ts_df_filepath, index_col="time", parse_dates=["time"])
lcd_ts_df.head()

,Log_1,Log_10,Log_101,Log_102,Log_111,Log_112,Log_113,Log_116,Log_117,Log_124,...,Log_8,Log_80,Log_82,Log_83,Log_86,Log_87,Log_89,Log_9,Log_98,Log_99
time,,,,,,,,,,,,,,,,,,,,,
2025-06-01 00:00:00,18.432771,17.569810,16.150085,16.733997,17.432288,17.386003,17.755843,17.137662,NaN,17.953002,...,17.955227,16.947178,16.886985,17.158579,17.394458,16.106470,15.643613,17.009486,15.816739,15.761997
2025-06-01 01:00:00,17.777206,17.289426,15.754876,16.280486,17.180832,16.786959,17.337937,16.371722,NaN,17.649475,...,17.750503,16.658783,16.511025,16.746903,17.130541,15.700802,16.089113,16.839030,15.749536,16.149640
2025-06-01 02:00:00,17.464777,17.393123,15.641387,16.381514,17.187508,16.990794,17.167925,16.314310,NaN,17.361524,...,17.957453,16.782063,16.303629,17.229343,17.476571,15.715489,16.029030,16.562206,16.507909,16.637198
2025-06-01 03:00:00,16.807876,17.359299,15.438443,16.270695,17.008151,16.369052,16.747794,15.977404,NaN,17.279634,...,17.847082,16.693497,16.013231,17.155019,17.184393,15.485618,15.585311,15.866585,16.346354,16.366382
2025-06-01 04:00:00,16.648991,16.980557,15.110437,15.977404,16.708184,16.292058,16.539953,15.549706,NaN,16.755805,...,17.493260,16.191030,15.875042,16.845038,16.863953,15.360113,15.554602,15.436217,16.044162,16.145634


In [ ]:
lcd_stations_gdf = gpd.read_file(lcd_stations_gdf_filepath).set_index("station_id")
lcd_stations_gdf.head()

,Location,Longitude,Latitude,LV_95_E,LV_95_N,Elev,valid,Tmean,Tmax,Tmin,Nmean,Dmean,TVar_All,TVar_Day,TVar_Night,TN,HD,WD,CDD,geometry
station_id,,,,,,,,,,,,,,,,,,,,
Log_1,VonRoll,7.42252,46.95291,2598773.471,1200203.276,553.9,92,21.156107,27.722226,15.281745,17.940964,22.787318,0.359814,0.405964,0.266984,7,34,17,300.808746,POINT (2598773.471 1200203.276)
Log_10,Hintere Länggasse,7.43541,46.95789,2599754.731,1200756.774,558.5,91,21.336500,27.266141,15.975521,18.425252,22.805452,0.548474,0.439871,0.764482,13,27,16,307.595208,POINT (2599754.731 1200756.774)
Log_101,Steinhölzliwald,7.42805,46.93514,2599194.143,1198227.698,573.7,67,18.584749,23.224271,14.063170,16.572313,19.708235,-1.644583,-2.111990,-0.712580,0,5,1,97.188200,POINT (2599194.143 1198227.698)
Log_102,Vordere Länggasse,7.43714,46.95171,2599886.391,1200069.733,561.6,92,21.061117,28.704617,15.225842,17.795112,22.712380,0.261263,0.330590,0.130492,5,40,16,296.336696,POINT (2599886.391 1200069.733)
Log_111,Tellstrasse,7.46146,46.96017,2601737.490,1201010.468,556.1,92,21.225882,27.819055,15.789339,18.343575,22.687705,0.428502,0.305916,0.678265,12,36,16,306.538451,POINT (2601737.49 1201010.468)


In [ ]:
aws_ts_cube = xr.open_dataset(aws_ts_cube_filepath).xvec.decode_cf()
aws_ts_cube

<xarray.Dataset> Size: 53kB
Dimensions:              (geometry: 1, time: 2208)
Coordinates:
  * geometry             (geometry) object 8B POINT (2601934 1204410)
  * time                 (time) datetime64[ns] 18kB 2025-06-01 ... 2025-08-31...
    station_id           (geometry) <U3 12B ...
Data variables:
    temperature          (time, geometry) float64 18kB ...
    radiation_shortwave  (time, geometry) float64 18kB ...
Indexes:
    geometry  GeometryIndex (crs=EPSG:2056)

In [ ]:
with open(station_model_dict_filepath) as src:
    model_filename = json.load(src)[station_model]
    model = io.load(path.join(models_dir, model_filename), trusted=SKOPS_TRUSTED)

In [ ]:
aws_rad_df = (
    aws_ts_cube["radiation_shortwave"]
    .set_index(geometry="station_id")
    .rename(geometry="station_id")
    .to_pandas()
    .dropna(how="all", axis="columns")
)
ref_station_gdf = aws_ts_cube[["geometry"]].xvec.to_geodataframe()
ref_station_gdf = ref_station_gdf[
    ref_station_gdf["station_id"].isin(aws_rad_df.columns)
]

# get a map of the nearest AWS for each LCD
nearest_aws_dict = lcd_stations_gdf.sjoin_nearest(ref_station_gdf)[
    "station_id_right"
].to_dict()

In [ ]:
yhat_df = pd.DataFrame(
    {
        column: model.predict(
            pd.DataFrame(
                {
                    "time": aws_rad_df.index,
                    "radiation_shortwave": aws_rad_df[nearest_aws_dict[column]].values,
                }
            )
        )
        for column in lcd_ts_df.columns
    },
    index=aws_rad_df.index,
)
yhat_df.head()

,Log_1,Log_10,Log_101,Log_102,Log_111,Log_112,Log_113,Log_116,Log_117,Log_124,...,Log_8,Log_80,Log_82,Log_83,Log_86,Log_87,Log_89,Log_9,Log_98,Log_99
time,,,,,,,,,,,,,,,,,,,,,
2025-06-01 00:00:00,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,...,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606
2025-06-01 01:00:00,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,...,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606
2025-06-01 02:00:00,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,...,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606,-0.344606
2025-06-01 03:00:00,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,...,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501,-0.353501
2025-06-01 04:00:00,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,...,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904,-0.189904


In [ ]:
lcd_ts_df.sub(yhat_df).dropna(how="all").to_csv(dst_ts_df_filepath)

## Correction performance (Zollikofen validation)

Evaluate each correction model by applying it back to the Zollikofen parallel
measurements (LCD vs. reference AWS `Prof`) and computing pre- and
post-correction agreement metrics.

In [ ]:
parallel_ts_df = (
    pd.read_csv(parallel_ts_df_filepath, parse_dates=["time"])
    .set_index("time")
    .resample("h")
    .mean()
)

ref_ts_df = (
    clients.MeteoSwissClient(validation_region)
    .get_ts_df(
        ["radiation_shortwave"],
        start=parallel_ts_df.index.min(),
        end=parallel_ts_df.index.max(),
    )
    .droplevel("station_id")
    .resample("h")
    .mean()
)

parallel_ts_df.head()

Stations:   0%|          | 0/1 [00:00<?, ?station/s]

,Prof,Abilium_2,Abilium_2_black,Decentlab,Barani,Koalasense,Onset_big,Onset_small
time,,,,,,,,
2025-07-27 00:00:00,15.533333,15.053025,NaN,NaN,15.200000,15.133333,15.334000,14.860000
2025-07-27 01:00:00,15.450000,14.956448,NaN,NaN,15.100000,14.983333,15.278667,14.700667
2025-07-27 02:00:00,15.366667,14.934641,NaN,NaN,15.083333,14.950000,15.191000,14.684833
2025-07-27 03:00:00,15.416667,14.929745,NaN,NaN,15.100000,14.983333,15.211000,14.704333
2025-07-27 04:00:00,14.883333,14.434399,NaN,NaN,14.616667,14.466667,14.907667,14.285833


In [ ]:
with open(station_model_dict_filepath) as src:
    all_models = {
        station_id: io.load(path.join(models_dir, filename), trusted=SKOPS_TRUSTED)
        for station_id, filename in json.load(src).items()
    }

station_models = [col for col in parallel_ts_df.columns if col != "Prof"]

records = []
for station_id in station_models:
    if station_id not in all_models:
        continue

    # ΔT = LCD − reference (Prof = Zollikofen MeteoSwiss station)
    delta_t = (parallel_ts_df[station_id] - parallel_ts_df["Prof"]).dropna()
    X_raw = pd.DataFrame(
        {
            "time": delta_t.index,
            "radiation_shortwave": delta_t.index.to_series().map(
                ref_ts_df["radiation_shortwave"]
            ),
        }
    )

    yhat = pd.Series(all_models[station_id].predict(X_raw), index=delta_t.index)
    residual = delta_t - yhat

    mae_pre = delta_t.abs().mean()
    rmse_pre = np.sqrt(np.mean(delta_t**2))
    mae_post = residual.abs().mean()
    rmse_post = np.sqrt(np.mean(residual**2))
    ss_res = np.sum(residual**2)
    ss_tot = np.sum((delta_t - delta_t.mean()) ** 2)

    records.append(
        {
            "LCD model": station_id,
            "R²": 1 - ss_res / ss_tot,
            "MAE [K]": mae_post,
            "RMSE [K]": rmse_post,
            "MAE reduction [K]": mae_pre - mae_post,
            "MAE reduction [%]": 100 * (mae_pre - mae_post) / mae_pre,
            "RMSE reduction [K]": rmse_pre - rmse_post,
            "RMSE reduction [%]": 100 * (rmse_pre - rmse_post) / rmse_pre,
        }
    )

correction_perf_df = pd.DataFrame(records)
display(correction_perf_df.round(4))

,LCD model,R²,MAE [K],RMSE [K],MAE reduction [K],MAE reduction [%],RMSE reduction [K],RMSE reduction [%]
0,Abilium_2,0.5907,0.2633,0.3486,0.1690,39.0924,0.2003,36.4887
1,Decentlab,0.7543,0.2309,0.3228,0.2962,56.1955,0.3514,52.1231
2,Barani,0.0975,0.2326,0.3088,0.0721,23.6613,0.0629,16.9267
3,Koalasense,0.3614,0.2058,0.2704,0.1146,35.7688,0.1166,30.1258
4,Onset_big,0.2136,0.2229,0.3041,0.0326,12.7456,0.0475,13.5136
